In [1]:
!pip install -q langchain langchain-groq gspread pydantic

import os
from google.colab import userdata, auth

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.2,
    max_retries=3
)

# Google authentication
auth.authenticate_user()

import gspread
from google.auth import default

creds, _ = default()
gc = gspread.authorize(creds)

# Open existing spreadsheet or create a new one
try:
    sh = gc.open("Sweet Crumbs Orders")
    print("Existing spreadsheet opened.")
except Exception:
    sh = gc.create("Sweet Crumbs Orders")
    print("New spreadsheet created.")

ws = sh.sheet1

# Add headers only if the sheet is empty
if not ws.get_all_values():
    ws.append_row(["name", "item", "qty", "date", "eggless", "total"])

print("Sheet ready!")
print(sh.url)

New spreadsheet created.
Sheet ready!
https://docs.google.com/spreadsheets/d/1X1XA7NGSZpaar4InN70_-n-JoRSUzn1xDl1kbnFfJXA


In [2]:
from typing import Literal
from pydantic import BaseModel, Field
from langchain.tools import tool

MENU = {
    "plum cake": 500,
    "chocolate cake": 650,
    "cupcake": 80
}

HOURS = {
    "monday": "9 am to 8 pm",
    "tuesday": "9 am to 8 pm",
    "wednesday": "9 am to 8 pm",
    "thursday": "9 am to 8 pm",
    "friday": "9 am to 8 pm",
    "saturday": "9 am to 8 pm",
    "sunday": "10 am to 6 pm"
}

class Order(BaseModel):
    name: str = Field(description="Customer name")
    item: Literal["plum cake", "chocolate cake", "cupcake"]
    qty: int = Field(ge=1)
    date: str = Field(description="Pickup date")
    eggless: bool = Field(description="Eggless preference")

In [3]:
@tool
def get_price(item: str, qty: int = 1) -> str:
    """Calculate the total price for a menu item and quantity."""

    item = item.lower().strip()
    price = MENU.get(item)

    if price is None:
        return f"Item not available. Menu: {', '.join(MENU)}"

    return f"{qty} x {item} = Rs {price * qty}"


@tool
def is_open(day: str) -> str:
    """Check bakery opening hours for a given day."""

    day = day.lower().strip()
    hours = HOURS.get(day)

    if hours:
        return f"Open {hours} on {day}."

    return "Invalid day. Please provide a weekday."


@tool
def log_order(name: str, item: str, qty: int, date: str, eggless: bool) -> str:
    """Save a complete customer order to Google Sheets."""

    try:
        order = Order(
            name=name,
            item=item.lower().strip(),
            qty=qty,
            date=date,
            eggless=eggless
        )

    except Exception as e:
        return f"Order validation failed: {e}"

    total = MENU[order.item] * order.qty

    ws.append_row([
        order.name,
        order.item,
        order.qty,
        order.date,
        "yes" if order.eggless else "no",
        total
    ])

    return (
        f"Order saved successfully! "
        f"{order.name}, {order.qty} {order.item}, "
        f"{order.date}, Eggless: {order.eggless}, "
        f"Total: Rs {total}"
    )

print("All tools are ready!")

All tools are ready!


In [4]:
from langchain.agents import create_agent

SYSTEM = """
ROLE:
You are a friendly order assistant for Sweet Crumbs Home Bakery.

FACTS:
Orders need 24 hours' notice.
All cakes have eggless versions.

MENU:
Plum cake: Rs 500
Chocolate cake: Rs 650
Cupcake: Rs 80

REQUIRED ORDER DETAILS:
1. Customer name
2. Item
3. Quantity
4. Pickup date
5. Eggless or regular preference

RULES:
- Always use get_price to calculate prices.
- Always use is_open to check timings when asked.
- Never guess prices or timings.
- Collect all five order details before saving.
- If a detail is missing, ask only for that missing detail.
- Remember details within the same customer conversation.
- Do not ask again for details already provided.
- Never save an incomplete order.
- Never assume quantity, item, date or eggless preference.
- Once all details are collected, call log_order exactly once.
- Never duplicate an order.
- After saving, confirm the order with its total.
- Do not mix details between customers or sessions.
- Orders require 24 hours' notice.

FORMAT:
Keep answers under 60 words.
Be friendly.
Reply in English or Telugu matching the customer.
"""

agent = create_agent(
    model=llm,
    tools=[get_price, is_open, log_order],
    system_prompt=SYSTEM
)

# Separate memory for every customer conversation
sessions = {}

def chat(session_id, text):
    if session_id not in sessions:
        sessions[session_id] = []

    history = sessions[session_id]

    try:
        result = agent.invoke({
            "messages": history + [
                {"role": "user", "content": text}
            ]
        })

        sessions[session_id] = result["messages"]

        return result["messages"][-1].content

    except Exception as e:
        return f"Agent error: {e}"

print("Sweet Crumbs AI Agent is ready!")

Sweet Crumbs AI Agent is ready!


In [5]:
import time

test_cases = [
    ("case1", [
        "Hi, I am Ravi. I want 2 eggless plum cakes for 24 December."
    ]),
    ("case2", [
        "Hi, I am Sravani. I need 5 regular cupcakes.",
        "I need them on 25 December."
    ]),
    ("case3", [
        "Hi, I am Kiran. I need eggless chocolate cakes on 26 December.",
        "I need 3 cakes."
    ]),
    ("case4", [
        "Hi, I am Anu. I need 3 eggless cakes for 26 December.",
        "I want plum cakes."
    ]),
    ("case5", [
        "I am Rahul. I need 4 regular cupcakes for 27 December."
    ]),
    ("case6", [
        "Hi, I am Priya. I want 1 eggless plum cake.",
        "28 December."
    ]),
    ("case7", [
        "I am Arjun. I want 2 regular chocolate cakes for 29 December."
    ])
]

for session_id, messages in test_cases:

    print(f"\n{'='*50}")
    print(f"TESTING {session_id.upper()}")
    print("="*50)

    for msg in messages:

        print(f"\nCustomer: {msg}")

        try:
            response = chat(session_id, msg)
            print("Assistant:", response)

        except Exception as e:
            print("Error:", e)

        time.sleep(2)

print("\nALL TEST CASES FINISHED")


TESTING CASE1

Customer: Hi, I am Ravi. I want 2 eggless plum cakes for 24 December.
Assistant: Order saved successfully! Ravi, 2 plum cake, 24 December, Eggless: True, Total: Rs 1000. Enjoy your sweet treat!

TESTING CASE2

Customer: Hi, I am Sravani. I need 5 regular cupcakes.
Assistant: Sure, Sravani! When would you like to pick up your 5 regular cupcakes?

Customer: I need them on 25 December.
Assistant: All set! Your 5 regular cupcakes are booked for 25 December. Total: Rs 400. Let me know if you need anything else. Enjoy!

TESTING CASE3

Customer: Hi, I am Kiran. I need eggless chocolate cakes on 26 December.
Assistant: Sure thing, Kiran! How many egg‑less chocolate cakes would you like to order?

Customer: I need 3 cakes.
Assistant: All set! Your 3 egg‑less chocolate cakes are booked for 26 December. Total Rs 1950. Let me know if you need anything else. Enjoy!

TESTING CASE4

Customer: Hi, I am Anu. I need 3 eggless cakes for 26 December.
Assistant: Sure, Anu! Which cake would 

In [6]:
print("Orders saved in Google Sheets:")
for row in ws.get_all_records():
    print(row)

print("\nSaved rows:", len(ws.get_all_records()))

Orders saved in Google Sheets:
{'Ravi': 'Sravani', 'plum cake': 'cupcake', '2': 5, '24 December': '25 December', 'yes': 'no', '1000': 400}
{'Ravi': 'Kiran', 'plum cake': 'chocolate cake', '2': 3, '24 December': '26 December', 'yes': 'yes', '1000': 1950}
{'Ravi': 'Anu', 'plum cake': 'plum cake', '2': 3, '24 December': '26 December', 'yes': 'yes', '1000': 1500}
{'Ravi': 'Rahul', 'plum cake': 'cupcake', '2': 4, '24 December': '27 December', 'yes': 'no', '1000': 320}

Saved rows: 4


In [7]:
records = ws.get_all_records()

print("Total saved orders:", len(records))

for row in records:
    print(row)

Total saved orders: 4
{'Ravi': 'Sravani', 'plum cake': 'cupcake', '2': 5, '24 December': '25 December', 'yes': 'no', '1000': 400}
{'Ravi': 'Kiran', 'plum cake': 'chocolate cake', '2': 3, '24 December': '26 December', 'yes': 'yes', '1000': 1950}
{'Ravi': 'Anu', 'plum cake': 'plum cake', '2': 3, '24 December': '26 December', 'yes': 'yes', '1000': 1500}
{'Ravi': 'Rahul', 'plum cake': 'cupcake', '2': 4, '24 December': '27 December', 'yes': 'no', '1000': 320}


In [8]:
print(chat(
    "case7",
    "I am Arjun. I want 2 regular chocolate cakes for 29 December."
))

Sure! Just let me know the day of the week for 29 December and whether you’d like the cakes eggless or regular. Once I have those, I’ll confirm the order and total.


In [9]:
records = ws.get_all_records()

print("Total saved orders:", len(records))

for row in records:
    print(row)

Total saved orders: 4
{'Ravi': 'Sravani', 'plum cake': 'cupcake', '2': 5, '24 December': '25 December', 'yes': 'no', '1000': 400}
{'Ravi': 'Kiran', 'plum cake': 'chocolate cake', '2': 3, '24 December': '26 December', 'yes': 'yes', '1000': 1950}
{'Ravi': 'Anu', 'plum cake': 'plum cake', '2': 3, '24 December': '26 December', 'yes': 'yes', '1000': 1500}
{'Ravi': 'Rahul', 'plum cake': 'cupcake', '2': 4, '24 December': '27 December', 'yes': 'no', '1000': 320}
